# 🎙️ Pipeline Tiền Xử Lý – VCTK Corpus

**Mục tiêu:** Lọc & chuẩn hóa 500 file `.wav` giọng nữ từ VCTK Corpus về đúng format:  
- Sample Rate: **16 kHz**  
- Độ dài: **5 giây** (cắt đầu / pad zero)  
- Output: `data/processed/<speaker>/<file>.wav`

| Bước | Nội dung |
|------|----------|
| 1 | Đọc `speaker-info.txt` → lọc female speakers |
| 2 | Duyệt toàn bộ `.wav` từ `wav48/<speaker>/` |
| 3 | Kiểm tra chất lượng (loại silence, corrupt, quá ngắn) |
| 4 | Resample 48 kHz → 16 kHz + Cắt/Pad → 5 giây |
| 5 | Sampling 500 file phân bổ đều theo speaker → lưu ra `data/processed/` |

## ⚙️ 0. Cấu hình (chỉnh tại đây nếu cần)

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# ─── Root project (Beta/) ─────────────────────────────────────────────────────
# notebooks nằm tại src/preprocessing/ → lên 2 cấp
PROJECT_ROOT = Path().resolve().parents[1]
load_dotenv(PROJECT_ROOT / ".env")
print(f"Project root: {PROJECT_ROOT}")

# ─── Nguồn dữ liệu VCTK ──────────────────────────────────────────────────────
VCTK_ROOT    = PROJECT_ROOT / "data" / "raw" / "VCTK-Corpus" / "VCTK-Corpus"
SPEAKER_INFO = VCTK_ROOT / "speaker-info.txt"
WAV48_DIR    = VCTK_ROOT / "wav48"

# ─── Output sau tiền xử lý ───────────────────────────────────────────────────
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" # data/processed/<speaker>/
FEATURES_DIR  = PROJECT_ROOT / "data" / "features" # data/features/<speaker>/

# ─── Tham số âm thanh ────────────────────────────────────────────────────────
TARGET_SR       = int(os.getenv("TARGET_SR", 16000))
TARGET_DURATION = float(os.getenv("TARGET_DURATION", 5.0))
TARGET_SAMPLES  = int(TARGET_SR * TARGET_DURATION)

MIN_DURATION = float(os.getenv("MIN_DURATION", 1.0))
MIN_RMS      = float(os.getenv("MIN_RMS", 1e-4))

# ─── Sampling ─────────────────────────────────────────────────────────────────
TOTAL_FILES = int(os.getenv("TOTAL_FILES", 1000))
RANDOM_SEED = int(os.getenv("RANDOM_SEED", 42))

print(f"  SPEAKER_INFO : {SPEAKER_INFO}")
print(f"  WAV48_DIR    : {WAV48_DIR}")

print(f"  TOTAL_FILES    : {TOTAL_FILES}")
print(f"  PROCESSED_DIR: {PROCESSED_DIR}")
print(f"  Target       : {TARGET_SAMPLES} samples ({TARGET_DURATION}s @ {TARGET_SR}Hz)")

Project root: C:\BTL_CSDLDPT\Beta
  SPEAKER_INFO : C:\BTL_CSDLDPT\Beta\data\raw\VCTK-Corpus\VCTK-Corpus\speaker-info.txt
  WAV48_DIR    : C:\BTL_CSDLDPT\Beta\data\raw\VCTK-Corpus\VCTK-Corpus\wav48
  TOTAL_FILES    : 1000
  PROCESSED_DIR: C:\BTL_CSDLDPT\Beta\data\processed
  Target       : 80000 samples (5.0s @ 16000Hz)


## 📦 Import thư viện

In [4]:
import random
import logging
from collections import defaultdict, Counter

import numpy as np
import librosa
import soundfile as sf
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
print("✅ Import thành công")

✅ Import thành công


---
## ① Đọc & lọc Female Speakers từ `speaker-info.txt`

File `speaker-info.txt` có định dạng:
```
ID   AGE  GENDER  ACCENTS  REGION
225   23     F    English  Southern England
226   22     M    English  Surrey
```
→ Lấy cột 3 (index 2) == `'F'`

In [5]:
def load_female_speakers(speaker_info_path: Path) -> list:
    """Trả về list speaker folder name của giọng nữ: ['p225', 'p228', ...]"""
    female_speakers = []
    with open(speaker_info_path, "r", encoding="utf-8") as f:
        next(f)  # bỏ header
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 3 and parts[2] == "F":
                female_speakers.append(f"p{parts[0]}")
    return female_speakers


female_speakers = load_female_speakers(SPEAKER_INFO)

print(f"✅ Tìm thấy {len(female_speakers)} female speakers")
print(f"   Ví dụ: {female_speakers[:8]} ...")

✅ Tìm thấy 61 female speakers
   Ví dụ: ['p225', 'p228', 'p229', 'p230', 'p231', 'p233', 'p234', 'p236'] ...


---
## ② Duyệt toàn bộ file `.wav` của Female Speakers

In [6]:
def collect_wav_files(wav48_dir: Path, speakers: list) -> dict:
    """Trả về dict: { 'p225': [Path, ...], 'p228': [...] }"""
    files_by_speaker = {}
    for speaker in speakers:
        speaker_dir = wav48_dir / speaker
        if not speaker_dir.exists():
            log.warning(f"Không tìm thấy thư mục: {speaker_dir}")
            continue
        wav_list = sorted(speaker_dir.glob("*.wav"))
        if wav_list:
            files_by_speaker[speaker] = wav_list
    return files_by_speaker


files_by_speaker = collect_wav_files(WAV48_DIR, female_speakers)

total_raw = sum(len(v) for v in files_by_speaker.values())
print(f"✅ Thu thập: {total_raw} file .wav từ {len(files_by_speaker)} speakers")
print(f"   Trung bình {total_raw / len(files_by_speaker):.0f} file/speaker")

✅ Thu thập: 25124 file .wav từ 61 speakers
   Trung bình 412 file/speaker


---
## ③ Kiểm tra chất lượng file

Loại bỏ các file:
| Lý do loại | Điều kiện |
|---|---|
| File bị corrupt | librosa raise exception |
| Quá ngắn | duration < 1 giây |
| Toàn silence | RMS ≈ 0 |

In [7]:
def is_valid_audio(wav_path: Path) -> bool:
    try:
        y, sr = librosa.load(str(wav_path), sr=None, mono=True)
        duration = len(y) / sr
        rms = float(np.sqrt(np.mean(y ** 2)))
        return duration >= MIN_DURATION and rms > MIN_RMS
    except Exception:
        return False


def filter_valid_files(files_by_speaker: dict) -> dict:
    all_files = [
        (spk, f)
        for spk, files in files_by_speaker.items()
        for f in files
    ]
    valid = defaultdict(list)
    rejected = 0
    for spk, wav_path in tqdm(all_files, desc="Kiểm tra chất lượng"):
        if is_valid_audio(wav_path):
            valid[spk].append(wav_path)
        else:
            rejected += 1
    log.info(f"Loại bỏ {rejected} file (lỗi / silence / quá ngắn)")
    return dict(valid)


valid_files = filter_valid_files(files_by_speaker)

total_valid = sum(len(v) for v in valid_files.values())
print(f"\n✅ Hợp lệ: {total_valid} / {total_raw} file")

Kiểm tra chất lượng:   0%|          | 0/25124 [00:00<?, ?it/s]

20:39:23 [INFO] Loại bỏ 3 file (lỗi / silence / quá ngắn)



✅ Hợp lệ: 25121 / 25124 file


---
## ④ Hàm Resample 48 kHz → 16 kHz & Cắt/Pad về 5 giây

```
File gốc (48kHz, N giây)
    → librosa.load(sr=16000)   # auto resample
    → cắt đầu 5s  hoặc  pad zero ở cuối
    → array (80000,) float32
```

In [8]:
def preprocess_audio(wav_path: Path) -> np.ndarray:
    """
    Load → resample → cắt/pad → ndarray (80000,) float32.
    librosa.load tự xử lý resample 48kHz → 16kHz.
    """
    y, _ = librosa.load(str(wav_path), sr=TARGET_SR, mono=True)

    if len(y) >= TARGET_SAMPLES:
        y = y[:TARGET_SAMPLES]                          # cắt lấy 5 giây đầu
    else:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)))     # pad zero ở cuối

    return y  # shape: (80_000,)


# ── Thử nghiệm nhanh với 1 file ──────────────────────────────────────────────
sample_speaker = next(iter(valid_files))
sample_wav     = valid_files[sample_speaker][0]
y_test         = preprocess_audio(sample_wav)

print(f"✅ Thử với: {sample_wav.name}")
print(f"   Output shape : {y_test.shape}  (mong đợi: ({TARGET_SAMPLES},))")
print(f"   dtype        : {y_test.dtype}")

✅ Thử với: p225_001.wav
   Output shape : (80000,)  (mong đợi: (80000,))
   dtype        : float32


---
## ⑤ Sampling 500 file – phân bổ đều theo speaker

> **Tại sao phân bổ đều?**  
> Nếu lấy ngẫu nhiên hoàn toàn, speaker nào có nhiều file sẽ chiếm đa số  
> → embedding bị kéo lệch về không gian của speaker đó.

In [9]:
def sample_files(valid_files_by_speaker: dict, total: int = TOTAL_FILES, seed: int = RANDOM_SEED) -> list:
    """
    Trả về list[(speaker, Path)] đủ `total` phần tử, phân bổ đều theo speaker.
    """
    random.seed(seed)
    n_speakers = len(valid_files_by_speaker)
    quota      = total // n_speakers  # ~7-8 file/speaker

    selected     = []
    leftover_pool = []

    for spk, files in valid_files_by_speaker.items():
        shuffled = files[:]
        random.shuffle(shuffled)
        selected.extend((spk, f) for f in shuffled[:quota])
        leftover_pool.extend((spk, f) for f in shuffled[quota:])

    # Bù nếu chưa đủ TOTAL_FILES
    deficit = total - len(selected)
    if deficit > 0 and leftover_pool:
        selected.extend(random.sample(leftover_pool, min(deficit, len(leftover_pool))))

    random.shuffle(selected)  # xáo trộn để không nhóm theo speaker
    return selected[:total]


selected = sample_files(valid_files)

counts = Counter(spk for spk, _ in selected)
print(f"✅ Đã chọn {len(selected)} file từ {len(counts)} speakers")
print(f"   File/speaker: min={min(counts.values())}, max={max(counts.values())}, avg={len(selected)/len(counts):.1f}")

✅ Đã chọn 1000 file từ 61 speakers
   File/speaker: min=16, max=19, avg=16.4


---
## 💾 Lưu file đã tiền xử lý ra `data/processed/<speaker>/`

In [10]:
def save_processed_files(selected: list, output_dir: Path) -> list:
    """
    Tiền xử lý từng file và lưu ra output_dir/<speaker>/<filename>.wav.
    Trả về list metadata record để dùng cho bước insert DB.

    Output structure (theo P9 Claude.md):
        data/processed/
        ├── p225/
        │   ├── p225_001.wav
        │   └── p225_002.wav
        └── p228/
            └── p228_005.wav
    """
    records = []

    for spk, src_path in tqdm(selected, desc="Xử lý & lưu file"):
        dest_dir  = output_dir / spk
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest_path = dest_dir / src_path.name

        if dest_path.exists():
            continue  # bỏ qua nếu đã xử lý, chạy lại an toàn

        try:
            y = preprocess_audio(src_path)
            sf.write(str(dest_path), y, TARGET_SR, subtype="PCM_16")  # PCM_16 tiết kiệm ~50% dung lượng

            records.append({
                "speaker"         : spk,
                "file_path"       : str(dest_path.relative_to(output_dir.parent)),
                "src_path"        : str(src_path),
                "sample_rate"     : TARGET_SR,
                "duration_sec"    : TARGET_DURATION,
            })
        except Exception as e:
            log.warning(f"Lỗi khi xử lý {src_path.name}: {e}")

    return records


records = save_processed_files(selected, PROCESSED_DIR)

print(f"\n✅ Đã lưu {len(records)} file vào {PROCESSED_DIR}")

Xử lý & lưu file:   0%|          | 0/1000 [00:00<?, ?it/s]


✅ Đã lưu 512 file vào C:\BTL_CSDLDPT\Beta\data\processed


---
## 📊 Báo cáo tổng kết

In [11]:
final_counts = Counter(r["speaker"] for r in records)

print("=" * 55)
print("  TỔNG KẾT TIỀN XỬ LÝ")
print("=" * 55)
print(f"  Tổng file đã xử lý  : {len(records)}")
print(f"  Số speakers          : {len(final_counts)}")
if final_counts:
    print(f"  File/speaker (min)   : {min(final_counts.values())}")
    print(f"  File/speaker (max)   : {max(final_counts.values())}")
    print(f"  File/speaker (avg)   : {len(records)/len(final_counts):.1f}")
print(f"  Output directory     : {PROCESSED_DIR}")
print(f"  Sample rate          : {TARGET_SR} Hz")
print(f"  Duration             : {TARGET_DURATION}s ({TARGET_SAMPLES} samples)")
print("=" * 55)
print("\n✅ Tiền xử lý hoàn tất! Sẵn sàng chạy feature extraction.")
print(f"   records[0] = {records[0] if records else 'N/A'}")

  TỔNG KẾT TIỀN XỬ LÝ
  Tổng file đã xử lý  : 512
  Số speakers          : 61
  File/speaker (min)   : 8
  File/speaker (max)   : 11
  File/speaker (avg)   : 8.4
  Output directory     : C:\BTL_CSDLDPT\Beta\data\processed
  Sample rate          : 16000 Hz
  Duration             : 5.0s (80000 samples)

✅ Tiền xử lý hoàn tất! Sẵn sàng chạy feature extraction.
   records[0] = {'speaker': 'p277', 'file_path': 'processed\\p277\\p277_349.wav', 'src_path': 'C:\\BTL_CSDLDPT\\Beta\\data\\raw\\VCTK-Corpus\\VCTK-Corpus\\wav48\\p277\\p277_349.wav', 'sample_rate': 16000, 'duration_sec': 5.0}
